In [2]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import RobertaTokenizer, RobertaForSequenceClassification, Trainer, TrainingArguments
import torch

/home/info-sec-lab/BTP/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Configuration
MODEL_NAME = "microsoft/codebert-base"
NUM_LABELS = 2
BATCH_SIZE = 8
LEARNING_RATE = 2e-5
EPOCHS = 3

class CodeDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

def load_and_preprocess_data(base_path, tokenizer, folder_name):
    codes = []
    labels = []
    for label_dir in ["Label_0", "Label_1"]:
        current_path = os.path.join(base_path, folder_name, label_dir)
        if not os.path.exists(current_path):
            print(f"Warning: Directory {current_path} not found. Skipping.")
            continue
        for filename in os.listdir(current_path):
            if filename.endswith(".txt"):
                filepath = os.path.join(current_path, filename)
                with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
                    codes.append(f.read())
                labels.append(0 if label_dir == "Label_0" else 1)

    # Tokenize the codes
    encodings = tokenizer(codes, truncation=True, padding=True, max_length=512)
    return CodeDataset(encodings, labels)



In [4]:
from transformers import RobertaModel


base_text_path = "../text_files/"

# Load tokenizer and model
tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)
model = RobertaForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)
model.eval()

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at microsoft/codebert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
         

In [5]:
# Load and preprocess training data
print("Loading training & validation data...")
train_dataset = load_and_preprocess_data(base_text_path, tokenizer, "train")
val_dataset = load_and_preprocess_data(base_text_path, tokenizer, "valid")

Loading training & validation data...


In [6]:
from transformers import Trainer, TrainingArguments
import numpy as np
from sklearn.metrics import accuracy_score

# -------------------- Compute Metrics --------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc}

# -------------------- Training Arguments --------------------
training_args = TrainingArguments(
    output_dir="../checkpoints/codebert_only",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    eval_strategy="epoch",             # Evaluate after every epoch
    save_strategy="epoch",
    load_best_model_at_end=True,       # Save best model based on metric
    metric_for_best_model="accuracy",
    greater_is_better=True
)

# -------------------- Trainer --------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,          # Validation dataset added
    compute_metrics=compute_metrics
)

# -------------------- Train --------------------
print("Training model with validation...")
trainer.train()

Training model with validation...


Epoch,Training Loss,Validation Loss,Accuracy
1,0.508400,0.499604,0.788830
2,0.336800,0.545792,0.798936
3,0.472200,0.637784,0.829255


TrainOutput(global_step=2820, training_loss=0.4274464626684257, metrics={'train_runtime': 1388.9798, 'train_samples_per_second': 16.242, 'train_steps_per_second': 2.03, 'total_flos': 5935785408921600.0, 'train_loss': 0.4274464626684257, 'epoch': 3.0})

In [7]:
import os
current_dir = os.getcwd()
print(current_dir)

/home/info-sec-lab/BTP/100k/experiments


In [9]:
from transformers import TrainingArguments, Trainer

# Specify the exact checkpoint to resume from
checkpoint_path = "../checkpoints/codebert_only/checkpoint-2820"
model = RobertaForSequenceClassification.from_pretrained(checkpoint_path)
tokenizer = RobertaTokenizer.from_pretrained("microsoft/codebert-base")

training_args = TrainingArguments(
    output_dir="../checkpoints/codebert_only",
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=5,  # Train for 5 more epochs
    warmup_steps=100,  # Reduced warmup since we're continuing
    weight_decay=0.01,
    logging_dir="./logs_resumed",
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    resume_from_checkpoint=checkpoint_path,  # Resume from specific checkpoint
    overwrite_output_dir=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# This will continue training from the checkpoint
trainer.train(resume_from_checkpoint=checkpoint_path)

/tmp/ipykernel_2480159/1878255467.py:26: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy
4,0.185200,0.735367,0.836170
5,0.227500,0.944914,0.827660


TrainOutput(global_step=4700, training_loss=0.07555841667062424, metrics={'train_runtime': 536.9272, 'train_samples_per_second': 70.028, 'train_steps_per_second': 8.754, 'total_flos': 9892975681536000.0, 'train_loss': 0.07555841667062424, 'epoch': 5.0})

## 🧪 Testing on Checkpoint 1 — `CodeBERT`


In [17]:
# Evaluate on test datasets
print("Evaluating on test datasets...")
for i in range(10):
    test_folder = f"Test_{i}"
    print(f"Loading test data for {test_folder}...")
    test_dataset = load_and_preprocess_data(base_text_path, tokenizer, test_folder)
    if len(test_dataset) > 0:
        predictions = trainer.predict(test_dataset)
        # Process predictions to get labels
        predicted_labels = predictions.predictions.argmax(axis=1)
        true_labels = test_dataset.labels

        from sklearn.metrics import accuracy_score, precision_recall_fscore_support
        accuracy = accuracy_score(true_labels, predicted_labels)
        precision, recall, f1, _ = precision_recall_fscore_support(true_labels, predicted_labels, average='binary')

        print(f"Results for {test_folder}:")
        print(f"  Accuracy: {accuracy:.4f}")
        print(f"  Precision: {precision:.4f}")
        print(f"  Recall: {recall:.4f}")
        print(f"  F1-Score: {f1:.4f}")
    else:
        print(f"No data found for {test_folder}. Skipping evaluation.")


Evaluating on test datasets...
Loading test data for Test_0...


Results for Test_0:
  Accuracy: 0.5050
  Precision: 0.5472
  Recall: 0.0579
  F1-Score: 0.1047
Loading test data for Test_1...


Results for Test_1:
  Accuracy: 0.5230
  Precision: 0.8286
  Recall: 0.0579
  F1-Score: 0.1082
Loading test data for Test_2...


Results for Test_2:
  Accuracy: 0.5291
  Precision: 0.8286
  Recall: 0.0579
  F1-Score: 0.1082
Loading test data for Test_3...


Results for Test_3:
  Accuracy: 0.5110
  Precision: 0.6170
  Recall: 0.0579
  F1-Score: 0.1058
Loading test data for Test_4...


...

Results for Test_5:
  Accuracy: 0.5130
  Precision: 0.6444
  Recall: 0.0579
  F1-Score: 0.1062
Loading test data for Test_6...


Results for Test_6:
  Accuracy: 0.5190
  Precision: 0.7436
  Recall: 0.0579
  F1-Score: 0.1074
Loading test data for Test_7...


Results for Test_7:
  Accuracy: 0.5150
  Precision: 0.6744
  Recall: 0.0579
  F1-Score: 0.1066
Loading test data for Test_8...


Results for Test_8:
  Accuracy: 0.5190
  Precision: 0.7436
  Recall: 0.0579
  F1-Score: 0.1074
Loading test data for Test_9...


Results for Test_9:
  Accuracy: 0.5150
  Precision: 0.6744
  Recall: 0.0579
  F1-Score: 0.1066


## 🧪 Testing on Checkpoint 3 — `CodeBERT`

In [19]:
# Evaluate on test datasets
print("Evaluating on test datasets...")
for i in range(10):
    test_folder = f"Test_{i}"
    print(f"Loading test data for {test_folder}...")
    test_dataset = load_and_preprocess_data(base_text_path, tokenizer, test_folder)
    if len(test_dataset) > 0:
        predictions = trainer.predict(test_dataset)
        # Process predictions to get labels
        predicted_labels = predictions.predictions.argmax(axis=1)
        true_labels = test_dataset.labels

        from sklearn.metrics import accuracy_score, precision_recall_fscore_support
        accuracy = accuracy_score(true_labels, predicted_labels)
        precision, recall, f1, _ = precision_recall_fscore_support(true_labels, predicted_labels, average='binary')

        print(f"Results for {test_folder}:")
        print(f"  Accuracy: {accuracy:.4f}")
        print(f"  Precision: {precision:.4f}")
        print(f"  Recall: {recall:.4f}")
        print(f"  F1-Score: {f1:.4f}")
    else:
        print(f"No data found for {test_folder}. Skipping evaluation.")


Evaluating on test datasets...
Loading test data for Test_0...


Results for Test_0:
  Accuracy: 0.5100
  Precision: 0.5625
  Recall: 0.0898
  F1-Score: 0.1549
Loading test data for Test_1...


Results for Test_1:
  Accuracy: 0.5299
  Precision: 0.7500
  Recall: 0.0898
  F1-Score: 0.1604
Loading test data for Test_2...


Results for Test_2:
  Accuracy: 0.5409
  Precision: 0.8182
  Recall: 0.0898
  F1-Score: 0.1619
Loading test data for Test_3...


Results for Test_3:
  Accuracy: 0.5070
  Precision: 0.5422
  Recall: 0.0898
  F1-Score: 0.1541
Loading test data for Test_4...


...

Results for Test_5:
  Accuracy: 0.5240
  Precision: 0.6818
  Recall: 0.0898
  F1-Score: 0.1587
Loading test data for Test_6...


Results for Test_6:
  Accuracy: 0.5279
  Precision: 0.7258
  Recall: 0.0898
  F1-Score: 0.1599
Loading test data for Test_7...


Results for Test_7:
  Accuracy: 0.5200
  Precision: 0.6429
  Recall: 0.0898
  F1-Score: 0.1576
Loading test data for Test_8...


Results for Test_8:
  Accuracy: 0.5279
  Precision: 0.7258
  Recall: 0.0898
  F1-Score: 0.1599
Loading test data for Test_9...


Results for Test_9:
  Accuracy: 0.5200
  Precision: 0.6429
  Recall: 0.0898
  F1-Score: 0.1576
